# LoRA GRPO Fine-Tuning with Training Hub (Interactive)

This notebook demonstrates how to run **GRPO (Group Relative Policy Optimization)** training directly inside a GPU-enabled workbench using Training Hub's ART backend. GRPO is a reinforcement learning technique that teaches a model to improve its outputs by comparing groups of responses and reinforcing the better ones.

## What is GRPO?

GRPO is a reinforcement learning from verifiable rewards (RLVR) algorithm that:
- Generates multiple candidate responses per prompt
- Scores them with a reward function (e.g. tool-call correctness)
- Uses the group's relative ranking to compute advantage signals
- Updates LoRA adapter weights via policy gradient with group normalization

Each training iteration has two phases:

1. **Rollout phase** — vLLM generates candidate responses and a reward function scores them
2. **Train phase** — Unsloth updates the LoRA adapter weights using the advantage signals

ART time-shares a single GPU between vLLM (inference) and Unsloth (training) via `gpu_memory_utilization`.

## Training Task: Tool-Call Verification

We use the [Agent-Ark/Toucan-1.5M](https://huggingface.co/datasets/Agent-Ark/Toucan-1.5M) dataset, which contains tool-calling conversations. The reward function verifies that the model produces syntactically correct tool calls with the expected function name and arguments.

## Hardware Requirements

| Resource | Minimum |
|----------|---------|
| **GPU** | 1× NVIDIA A100, H100, or L40S (40GB+ VRAM) |
| **CPU** | 8 cores |
| **Memory** | 64Gi |

- ART manages GPU memory sharing between vLLM and Unsloth via `gpu_memory_utilization`
- The default `gpu_memory_utilization=0.45` reserves 45% for vLLM, leaving the rest for Unsloth
- CPU and memory requirements are high because ART spawns multiple processes (vLLM engine + Unsloth training) alongside the Jupyter server

> **Note:** This notebook runs training directly inside the workbench. For job submission via Kubeflow Trainer, see [`grpo_lora-kubeflow-trainjob.ipynb`](./grpo_lora-kubeflow-trainjob.ipynb).

## Setup

First, import the required dependencies.

In [ ]:
import json
from pathlib import Path

import torch

In [ ]:
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_memory:.1f} GB")
else:
    print("WARNING: No GPU detected. GRPO requires a GPU with 40GB+ VRAM.")

## 1. Explore the Dataset

We use the [Agent-Ark/Toucan-1.5M](https://huggingface.co/datasets/Agent-Ark/Toucan-1.5M) dataset with the `Qwen3` configuration. This dataset contains multi-turn tool-calling conversations where the model must produce syntactically correct function calls.

ART's built-in reward function handles scoring automatically — it checks that the model output is a valid tool call with the correct function name and arguments.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Agent-Ark/Toucan-1.5M", "Qwen3", split="train", streaming=True)

print("Dataset: Agent-Ark/Toucan-1.5M (Qwen3 config)")
print("=" * 60)

sample = next(iter(dataset))
print(f"\nColumns: {list(sample.keys())}")

messages = sample["messages"]
if isinstance(messages, str):
    messages = json.loads(messages)

print(f"\nSample conversation ({len(messages)} turns):")
print("-" * 60)
for msg in messages[:4]:
    role = msg["role"]
    content = (
        msg["content"][:200] + "..." if len(msg["content"]) > 200 else msg["content"]
    )
    print(f"\n[{role}]\n{content}")

## 2. Configure Training Parameters

Key parameters:

### GRPO Parameters
- **num_iterations**: Number of GRPO iterations (each = rollout + train phase)
- **group_size**: Responses generated per prompt for comparison
- **prompt_batch_size**: Unique prompts sampled per iteration

### LoRA Parameters
- **lora_r**: Rank of the low-rank matrices (higher = more capacity, more memory)
- **lora_alpha**: Scaling factor

### vLLM Parameters
- **gpu_memory_utilization**: Fraction of GPU memory reserved for vLLM inference (rest for Unsloth training)

In [ ]:
# Model and dataset
MODEL_PATH = "Qwen/Qwen3-4B"
DATA_PATH = "Agent-Ark/Toucan-1.5M"
DATA_CONFIG = "Qwen3"

# Output directory (local to workbench)
OUTPUT_DIR = Path("./grpo_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# GRPO hyperparameters
NUM_ITERATIONS = 5  # Number of GRPO iterations (each = rollout + train)
GROUP_SIZE = 4  # Responses generated per prompt for comparison
PROMPT_BATCH_SIZE = 50  # Unique prompts per iteration
N_TRAIN = 200  # Total training examples from dataset
LEARNING_RATE = 1e-5

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 8

# vLLM configuration — controls GPU memory split between vLLM and Unsloth
GPU_MEMORY_UTILIZATION = 0.45

print("Training Configuration:")
print(f"  Model: {MODEL_PATH}")
print(f"  Dataset: {DATA_PATH} ({DATA_CONFIG})")
print(f"  Output: {OUTPUT_DIR.resolve()}")
print(f"  Iterations: {NUM_ITERATIONS}")
print(f"  Group size: {GROUP_SIZE}")
print(f"  Prompts/iteration: {PROMPT_BATCH_SIZE}")
print(f"  Training examples: {N_TRAIN}")
print(f"  LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"  GPU memory for vLLM: {GPU_MEMORY_UTILIZATION * 100:.0f}%")

## 3. Run GRPO Training

Call `training_hub.lora_grpo()` directly. ART manages the single-GPU workflow internally:
1. Loads the base model with LoRA adapters via Unsloth
2. Starts a vLLM inference process for rollout generation
3. Alternates between rollout (generate + score) and train (update weights) phases

> **Note:** Training may take 10-30 minutes depending on configuration.

In [ ]:
from training_hub import lora_grpo

result = lora_grpo(
    model_path=MODEL_PATH,
    data_path=DATA_PATH,
    data_config=DATA_CONFIG,
    ckpt_output_dir=str(OUTPUT_DIR),
    backend="art",
    # GRPO hyperparameters
    num_iterations=NUM_ITERATIONS,
    group_size=GROUP_SIZE,
    prompt_batch_size=PROMPT_BATCH_SIZE,
    n_train=N_TRAIN,
    learning_rate=LEARNING_RATE,
    # LoRA
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    # vLLM
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
)

print(f"\nTraining completed — status: {result.get('status')}")

## 4. Inspect Training Metrics

GRPO writes two lines per iteration to `training_metrics.jsonl`:
- **Rollout phase**: `mean_reward`, `full_match_rate`
- **Train phase**: `loss`, `grad_norm`, `entropy`

In [ ]:
metrics_file = OUTPUT_DIR / "training_metrics.jsonl"

if metrics_file.exists():
    print("Training metrics:")
    print("=" * 80)
    with open(metrics_file) as f:
        for line in f:
            entry = json.loads(line)
            phase = entry.get("phase", "unknown")
            step = entry.get("step", "?")
            if phase == "rollout":
                print(
                    f"Step {step} [rollout]  mean_reward={entry.get('mean_reward', 'N/A'):.4f}  "
                    f"full_match_rate={entry.get('full_match_rate', 'N/A'):.4f}"
                )
            elif phase == "train":
                print(
                    f"Step {step} [train]    loss={entry.get('loss', 'N/A')}  "
                    f"grad_norm={entry.get('grad_norm', 'N/A')}  "
                    f"entropy={entry.get('entropy', 'N/A')}"
                )
else:
    print("No metrics file found — training may not have completed.")

In [ ]:
results_file = OUTPUT_DIR / "training_results.json"

if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)
    print("Training results:")
    print(json.dumps(results, indent=2, default=str))
else:
    print("No results file yet — training may not have completed.")

## 5. Plot Reward Curve

Visualize how the model's reward improves over GRPO iterations.

In [ ]:
metrics_file = OUTPUT_DIR / "training_metrics.jsonl"

if not metrics_file.exists():
    print("No metrics file found.")
else:
    steps, rewards, match_rates = [], [], []
    with open(metrics_file) as f:
        for line in f:
            entry = json.loads(line)
            if entry.get("phase") == "rollout":
                steps.append(entry["step"])
                rewards.append(entry["mean_reward"])
                match_rates.append(entry["full_match_rate"])

    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(steps, rewards, marker="o")
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Mean Reward")
    ax1.set_title("GRPO Training: Mean Reward")
    ax1.grid(True, alpha=0.3)

    ax2.plot(steps, match_rates, marker="o", color="tab:orange")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Full Match Rate")
    ax2.set_title("GRPO Training: Full Match Rate")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "reward_curve.png", dpi=150)
    plt.show()
    print(f"Plot saved to {OUTPUT_DIR}/reward_curve.png")

## 6. Test the Trained Model

Load the fine-tuned LoRA adapter and test whether the model generates better tool calls after GRPO training.

In [ ]:
from unsloth import FastLanguageModel

# ART saves checkpoints under .art/<project>/models/<name>/checkpoints/<step>/
# Find the latest checkpoint dynamically.
art_dir = OUTPUT_DIR / ".art"
checkpoints_dirs = sorted(art_dir.rglob("checkpoints"))
if not checkpoints_dirs:
    raise FileNotFoundError(f"No checkpoints found under {art_dir}")
latest_ckpt = sorted(checkpoints_dirs[0].iterdir())[-1]
print(f"Loading checkpoint: {latest_ckpt}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(latest_ckpt),
    max_seq_length=2048,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)

print("Fine-tuned model loaded and ready for inference.")

In [ ]:
def generate_response(messages, max_tokens=512):
    """
    Generate a response from the fine-tuned model.
    """
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    return response

In [ ]:
test_prompts = [
    {
        "description": "Weather lookup",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant with access to tools. "
                    "Available tools: get_weather(location: str, unit: str = 'celsius') -> dict"
                ),
            },
            {"role": "user", "content": "What's the weather like in Dublin?"},
        ],
    },
    {
        "description": "Calculator",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant with access to tools. "
                    "Available tools: calculate(expression: str) -> float"
                ),
            },
            {"role": "user", "content": "What is 1547 * 23 + 89?"},
        ],
    },
    {
        "description": "File search",
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant with access to tools. "
                    "Available tools: search_files(query: str, directory: str = '.', file_type: str = None) -> list"
                ),
            },
            {
                "role": "user",
                "content": "Find all Python files that contain 'import torch'",
            },
        ],
    },
]

print("Testing fine-tuned model on tool-call generation:")
print("=" * 60)

for test in test_prompts:
    print(f"\nTest: {test['description']}")
    print(f"User: {test['messages'][-1]['content']}")
    response = generate_response(test["messages"])
    print(f"Model: {response}")
    print("-" * 60)

## 7. Save the Model (Optional)

The LoRA checkpoint is already saved in the output directory. You can inspect the saved files or reload the model later.

In [ ]:
print("Training artifacts:")
for file in sorted(OUTPUT_DIR.rglob("*")):
    if file.is_file() and ".ipynb_checkpoints" not in str(file):
        size_mb = file.stat().st_size / (1024 * 1024)
        rel = file.relative_to(OUTPUT_DIR)
        print(f"  {rel}: {size_mb:.2f} MB")

In [ ]:
# To reload the model later, find the latest checkpoint:
#
# from pathlib import Path
# from unsloth import FastLanguageModel
#
# art_dir = Path("./grpo_output/.art")
# latest_ckpt = sorted(sorted(art_dir.rglob("checkpoints"))[0].iterdir())[-1]
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=str(latest_ckpt),
#     max_seq_length=2048,
#     load_in_4bit=False,
# )
# FastLanguageModel.for_inference(model)

## Summary

In this notebook, we:

1. **Explored** the Agent-Ark/Toucan-1.5M tool-calling dataset
2. **Configured** GRPO hyperparameters and LoRA settings
3. **Trained** a LoRA adapter using GRPO with Training Hub's ART backend
4. **Inspected** training metrics (reward curve, match rates)
5. **Tested** the fine-tuned model on tool-call generation examples

### Key Takeaways

- **GRPO** improves model outputs through reinforcement learning from verifiable rewards
- **ART** efficiently time-shares a single GPU between vLLM inference and Unsloth training
- The `gpu_memory_utilization` parameter controls the memory split between inference and training
- Training Hub handles the full GRPO loop: rollout generation, reward scoring, and weight updates

### Next Steps

- Increase `num_iterations` and `n_train` for better results (more compute time)
- Experiment with `group_size` (larger groups give better advantage estimates but use more memory)
- For distributed or long-running training, use the [TrainJob notebook](./grpo_lora-kubeflow-trainjob.ipynb)
- Add W&B logging for experiment tracking